In [4]:
from pathlib import Path

import pandas as pd

# File name
SMOTE_DATA_FILE_NAME = "loan_data_smoted_raw_train_4.csv"
RAW_DATA_FILE_NAME = "loan_data_raw_test_1.csv"
PROJECT_ROOT = Path("..")
SMOTE_DATA_FILE_PATH = PROJECT_ROOT / "data" / "smote" / SMOTE_DATA_FILE_NAME
RAW_DATA_FILE_PATH = PROJECT_ROOT / "data" / "raw" / RAW_DATA_FILE_NAME
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Smote data path:", SMOTE_DATA_FILE_PATH)
print("Raw data path:", RAW_DATA_FILE_PATH)

train_df = pd.read_csv(SMOTE_DATA_FILE_PATH)
test_df = pd.read_csv(RAW_DATA_FILE_PATH)


PROJECT_ROOT: ..
Smote data path: ..\data\smote\loan_data_smoted_raw_train_4.csv
Raw data path: ..\data\raw\loan_data_raw_test_1.csv


In [9]:
# Labels
target = "loan_status"

X_train = train_df.drop(columns=[target]).copy()
y_train = train_df[target].copy()
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)


X_test = test_df.drop(columns=[target])
y_test = test_df[target].copy()
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (53530, 12)
y_train shape: (53530,)
X_test shape: (8999, 12)
y_test shape: (8999,)


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Numeric columns includes person_education 
numeric_cols = [
    "person_age", "person_income", "person_emp_exp", "loan_amnt",
    "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length", 
    "credit_score", "person_education" 
]

cat_cols = ["person_gender", "person_home_ownership", "loan_intent"]

col_names = numeric_cols + cat_cols
num_features = len(numeric_cols) + len(cat_cols)

# Scale numeric columns and convert categorical columns to one hot
# Possible issue of using categorical data in PCA
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('nom', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
    ]
)

# Pipiline includes preprocessing, PCA, and logistic regression modelling
pipeline = Pipeline([
    ('prep', preprocessor),
    ('pca', PCA(n_components=0.95)),
    ('clf', LogisticRegression(max_iter=1000))
])

# Fit and predict
pipeline.fit(X_train, y_train)
y_test_pred_PCA = pipeline.predict(X_test)
y_train_pred_PCA = pipeline.predict(X_train)
print(f"PCA + logistic regression training accuracy: {accuracy_score(y_train, y_train_pred_PCA)*100:.2f}%")
print(f"PCA + logistic regressin test accuracy: {accuracy_score(y_test, y_test_pred_PCA)*100:.2f}%")

PCA + logistic regression training accuracy: 78.29%
PCA + logistic regressin test accuracy: 75.93%
